# 7. Punktskattning och konfidensintervall

I [notebook 5](../05_population_och_stickprov/05_population_och_stickprov.ipynb) såg du att ett stickprovsmedelvärde
aldrig träffar μ exakt, men att vi kan räkna ut hur mycket det brukar variera: **medelfelet**.
Nu använder vi det för att ge ett ärligt svar. I stället för "medellängden är 170,6 cm" säger vi
"medellängden är troligen mellan 168,9 och 172,2 cm".

**Efter den här notebooken kan du**
- skilja på punktskattning och intervallskattning
- räkna ut ett konfidensintervall för ett medelvärde och för en andel
- förklara hur stickprovsstorlek och konfidensgrad påverkar intervallets bredd
- tolka ett konfidensintervall korrekt, och känna igen den vanligaste feltolkningen
- förklara varför t-fördelningen används

**Förkunskaper:** [5. Population och stickprov](../05_population_och_stickprov/05_population_och_stickprov.ipynb)
och [4. Fördelningar](../04_fordelningar/04_fordelningar.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

N = 10_000
population = pd.DataFrame({
    "height": rng.normal(172, 9, N).round(1),                       # cm
    "num_children": rng.poisson(1.5, N),
    "education": rng.choice(["Grundskola", "Gymnasium", "Högskola"], N, p=[0.2, 0.45, 0.35]),
    "income": rng.lognormal(np.log(32), 0.4, N).round(1),           # tkr per månad
})
heights = population["height"].to_numpy()
mu = population["height"].mean()                                 # sanningen – okänd i verkligheten
p = (population["education"] == "Högskola").mean()

sample = population.sample(n=100, random_state=1)                # vårt enda stickprov
height = sample["height"]

## 7.1 Idén: skattning ± felmarginal

En **punktskattning** är ett enda tal, till exempel x̄ som skattning av μ. Den säger ingenting om hur säker den är.

Från notebook 5 vet vi två saker om x̄:
1. x̄ är ungefär **normalfördelat** kring μ (centrala gränsvärdessatsen).
2. Dess standardavvikelse är **medelfelet** $s/\sqrt{n}$.

Från notebook 4 vet vi att 95 % av en normalfördelning ligger inom ungefär 2 standardavvikelser från mitten. Alltså:

> I 95 % av alla stickprov hamnar x̄ högst 2 medelfel från μ.
> Då ligger också μ högst 2 medelfel från x̄.

Det är hela idén. Ett **konfidensintervall** är *skattning ± felmarginal*, där felmarginalen är ungefär 2 medelfel.

In [ ]:
n = len(height)
x_bar = height.mean()
s = height.std()
std_error = s / np.sqrt(n)

print(f"Punktskattning: x̄ = {x_bar:.2f} cm")
print(f"Medelfel:       {std_error:.2f} cm")
print(f"Grovt 95 % KI:  {x_bar - 2 * std_error:.1f} till {x_bar + 2 * std_error:.1f} cm")

## 7.2 Den exakta formeln

Talet 2 var en avrundning. Den exakta formeln är

$$\text{KI} = \bar{x} \pm \underbrace{t^* \cdot \frac{s}{\sqrt{n}}}_{\text{felmarginal}}$$

där $t^*$ hämtas från **t-fördelningen** med $n - 1$ **frihetsgrader**. För ett 95 % intervall ska 2,5 % ligga i varje svans,
så vi slår upp värdet som har 97,5 % under sig. (Varför t-fördelningen och inte normalfördelningen? Det kommer i avsnitt 7.6.)

In [ ]:
t_crit = stats.t.ppf(0.975, df=n - 1)
margin_of_error = t_crit * std_error

print(f"t* = {t_crit:.3f}  (frihetsgrader = {n - 1})")
print(f"Felmarginal: ±{margin_of_error:.2f} cm")
print(f"95 % KI: [{x_bar - margin_of_error:.2f}, {x_bar + margin_of_error:.2f}]")

scipy kan göra hela beräkningen i ett steg:

In [ ]:
stats.t.interval(0.95, df=n - 1, loc=x_bar, scale=std_error)

Eftersom vi har facit kan vi kontrollera: täcker intervallet det sanna μ?

In [ ]:
print(f"Sant μ = {mu:.2f}")

## 7.3 Vad gör intervallet brett eller smalt?

Felmarginalen $t^* \cdot s/\sqrt{n}$ har tre delar. Funktionen nedan räknar ut ett konfidensintervall så att du kan
experimentera med två av dem.

In [ ]:
def ci_mean(data, level=0.95):
    """Konfidensintervall för ett medelvärde. Returnerar (nedre, övre)."""
    n = len(data)
    std_error = np.std(data, ddof=1) / np.sqrt(n)
    t_crit = stats.t.ppf(1 - (1 - level) / 2, df=n - 1)
    return np.mean(data) - t_crit * std_error, np.mean(data) + t_crit * std_error

🔮 **Gissa först:** blir ett 99 % intervall bredare eller smalare än ett 95 % intervall?
Och vad händer med bredden om stickprovet görs fyra gånger så stort?

In [ ]:
sample_size = 100       # ← ändra: pröva 25, 100, 400 och 1600
level = 0.95            # ← ändra: pröva 0.80, 0.95 och 0.99

data = population["height"].sample(sample_size, random_state=1)
low, high = ci_mean(data, level)
print(f"n = {sample_size}, konfidensgrad {level:.0%}: [{low:.2f}, {high:.2f}]   bredd {high - low:.2f} cm")

🧪 **Labba:** Ändra en sak i taget och anteckna bredden.

<details>
<summary>💡 <b>Vad bör du ha sett?</b></summary>

- **Större n → smalare intervall.** Fyra gånger så stort stickprov ger ungefär halva bredden ($\sqrt{n}$ igen).
- **Högre konfidensgrad → bredare intervall.** Vill man vara säkrare på att fånga μ måste man ta i med ett bredare nät.
  Ett 100 % intervall skulle vara oändligt brett och helt värdelöst.
- Den tredje delen, spridningen *s*, kan vi inte välja: brokigare data ger bredare intervall.

</details>

✏️ **Din tur:** Räkna ut ett 99 % konfidensintervall för medellängden i `height` **steg för steg**, utan `ci_mean`.
Vilket tal ska ersätta `0.975` i `stats.t.ppf`? Kontrollera sedan med `ci_mean(height, 0.99)`.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
t_crit_99 = stats.t.ppf(0.995, df=n - 1)       # 0,5 % i varje svans
print(f"t* = {t_crit_99:.3f}")
print(f"99 % KI: [{x_bar - t_crit_99 * std_error:.2f}, {x_bar + t_crit_99 * std_error:.2f}]")
print("Kontroll:", ci_mean(height, 0.99))
```

</details>

## 7.4 Hur ska "95 %" tolkas?

Det här är det svåraste i notebooken, och det enklaste sättet att förstå det är att simulera.
Vi drar 100 nya stickprov, räknar ut ett intervall från vart och ett och ser hur många som täcker det sanna μ.

🔮 **Gissa först:** hur många av 100 stycken 95 %-intervall kommer att missa μ?

In [ ]:
level = 0.95            # ← ändra: pröva 0.80 och 0.99
sample_size = 30        # ← ändra: pröva 10 och 200

fig, ax = plt.subplots(figsize=(10, 4))
misses = 0
for i in range(100):
    one_sample = rng.choice(heights, sample_size, replace=False)
    low, high = ci_mean(one_sample, level)
    hit = low <= mu <= high
    misses += not hit
    ax.plot([i, i], [low, high], color="tab:blue" if hit else "red")
ax.axhline(mu, color="black", linestyle="--", label="sant μ")
ax.set_title(f"100 konfidensintervall ({level:.0%}), n = {sample_size}: {misses} missar μ (röda)")
ax.set_xlabel("Stickprov nummer")
ax.set_ylabel("Längd (cm)")
ax.legend()
plt.show()

🧪 **Labba:**
1. Kör cellen flera gånger. Antalet missar varierar, men kring vilket tal?
2. Sätt `level = 0.80`. Hur många missar nu? Vad hände med intervallens längd?
3. Sätt tillbaka 0.95 och ändra `sample_size` till 200. Ändras *andelen* missar? Vad ändras i stället?

**Tolkningen:** om undersökningen upprepas många gånger kommer ungefär 95 % av intervallen att täcka det sanna μ.
Det är alltså **metoden** som träffar rätt i 95 % av fallen. Lägg märke till att μ (den streckade linjen) ligger still.
Det är intervallen som hoppar.

> **Vanlig feltolkning:** "Det är 95 % sannolikhet att μ ligger i just det här intervallet." Men μ är ett fast tal,
> och ett färdigräknat intervall antingen täcker μ eller gör det inte. Vi vet bara inte vilket.
> Varje rött streck i figuren är ett stickprov där forskaren gjorde allt rätt och ändå fick ett intervall som missar.

Stickprovsstorleken ändrar inte andelen träffar, bara hur **smala** intervallen är. Ett 95 % intervall från n = 10 är lika
pålitligt som ett från n = 1000, men mycket mindre användbart.

✏️ **Din tur:** 100 intervall är för få för att se om det verkligen är 95 %. Skriv en loop som drar 5 000 stickprov
med n = 30 och räknar *andelen* intervall som täcker μ.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
hits = 0
num_trials = 5000
for _ in range(num_trials):
    low, high = ci_mean(rng.choice(heights, 30, replace=False), 0.95)
    hits += low <= mu <= high
print(f"Andel intervall som täcker μ: {hits / num_trials:.1%}")
```

</details>

## 7.5 Konfidensintervall för en andel

Samma idé, med medelfelet för en andel:

$$\hat{p} \pm z^* \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

Här används normalfördelningens $z^*$ (1,96 för 95 %). **Förutsättning:** stickprovet ska vara så stort att
normalfördelningen passar. Tumregel: $n \cdot \hat{p}(1-\hat{p}) > 5$.

In [ ]:
p_hat = (sample["education"] == "Högskola").mean()
std_error_p = np.sqrt(p_hat * (1 - p_hat) / n)
z_crit = stats.norm.ppf(0.975)

print(f"p̂ = {p_hat:.2f}   tumregel: n·p̂(1−p̂) = {n * p_hat * (1 - p_hat):.1f}")
print(f"Felmarginal: ±{z_crit * std_error_p:.3f}")
print(f"95 % KI: [{p_hat - z_crit * std_error_p:.3f}, {p_hat + z_crit * std_error_p:.3f}]   (sant p = {p:.3f})")

Med n = 100 är felmarginalen ungefär ±9 procentenheter. Intervallet täcker det sanna värdet, men det är så brett att
det inte säger särskilt mycket.

✏️ **Din tur:** En opinionsmätning frågar 1 500 personer, och 6,0 % säger att de skulle rösta på ett visst parti.
1. Räkna ut ett 95 % konfidensintervall.
2. Spärren till riksdagen är 4 %. Kan man vara ganska säker på att partiet ligger över spärren?
3. Månaden efter får partiet 6,8 %. Tidningen skriver "Partiet ökar!". Vad säger du?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
p_hat_poll, n_poll = 0.060, 1500
moe = stats.norm.ppf(0.975) * np.sqrt(p_hat_poll * (1 - p_hat_poll) / n_poll)
print(f"Felmarginal: ±{moe:.3f}")
print(f"95 % KI: [{p_hat_poll - moe:.3f}, {p_hat_poll + moe:.3f}]")
# 2. Ja: hela intervallet (ungefär 4,8–7,2 %) ligger över 4 %.
# 3. 6,8 % ligger inom felmarginalen för den första mätningen, och den nya mätningen har sin egen osäkerhet.
#    Förändringen kan mycket väl vara slump.
```

</details>

## 7.6 Varför t-fördelningen?

Formeln använder *s* i stället för det okända σ. Men *s* är också en skattning och varierar mellan stickprov,
särskilt när stickprovet är litet. Den extra osäkerheten gör att vi måste ta i lite mer.
**t-fördelningen** är en normalfördelning med **tjockare svansar** som kompenserar för detta.

Hur mycket tjockare beror på antalet **frihetsgrader**, n − 1.

In [ ]:
x = np.linspace(-4, 4, 300)
plt.plot(x, stats.norm.pdf(x), "k--", label="Normalfördelning")
for dof in [1, 5, 30]:
    plt.plot(x, stats.t.pdf(x, df=dof), label=f"t, {dof} frihetsgrader")
plt.xlabel("Värde")
plt.ylabel("Täthet")
plt.title("t-fördelningen för olika frihetsgrader")
plt.legend()
plt.show()

✏️ **Din tur:** Skriv ut `stats.t.ppf(0.975, df=dof)` för 2, 5, 10, 30, 100 och 1000 frihetsgrader, och jämför med
`stats.norm.ppf(0.975)`. Vad händer med t* när stickprovet växer? Från ungefär vilket n spelar skillnaden ingen större roll?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
for dof in [2, 5, 10, 30, 100, 1000]:
    print(f"frihetsgrader = {dof:>4}: t* = {stats.t.ppf(0.975, df=dof):.3f}")
print(f"normalfördelning:     z* = {stats.norm.ppf(0.975):.3f}")
# t* närmar sig 1,96 när antalet frihetsgrader växer. Redan vid 30 är skillnaden liten.
```

</details>

Vad händer om man struntar i detta och använder 1,96 även för små stickprov? Det kan vi testa.
Koden drar 5 000 stickprov med n = 5 och räknar hur ofta intervallen täcker μ:

In [ ]:
sample_size = 5        # ← ändra: pröva 5, 15 och 50

covered_z = covered_t = 0
for _ in range(5000):
    one_sample = rng.choice(heights, sample_size, replace=False)
    m = one_sample.mean()
    se = one_sample.std(ddof=1) / np.sqrt(sample_size)
    covered_z += abs(m - mu) <= 1.96 * se
    covered_t += abs(m - mu) <= stats.t.ppf(0.975, df=sample_size - 1) * se

print(f"n = {sample_size}")
print(f"Täckning med z* = 1,96: {covered_z / 5000:.1%}")
print(f"Täckning med t*:        {covered_t / 5000:.1%}")

Med n = 5 och 1,96 är "95 %-intervallet" i själva verket bara ett 88 %-intervall. t-fördelningen ger det vi lovar.
Med större stickprov försvinner skillnaden.

## 7.7 Förutsättningar

Ett konfidensintervall för ett medelvärde bygger på att
1. stickprovet är **slumpmässigt** (annars kan det finnas bias, se [notebook 6](../06_datakvalitet_och_bias/06_datakvalitet_och_bias.ipynb)), och
2. populationen är ungefär normalfördelad **eller** stickprovet är tillräckligt stort för centrala gränsvärdessatsen (tumregel n ≥ 30).

Intervallet mäter bara det **slumpmässiga** urvalsfelet. En snedvriden undersökning ger ett smalt och prydligt intervall
kring fel värde.

## Sammanfattning

- **Konfidensintervall = punktskattning ± felmarginal**, där felmarginalen är ett kritiskt värde gånger medelfelet.
- Medelvärde: $\bar{x} \pm t^* \cdot s/\sqrt{n}$. Andel: $\hat{p} \pm z^* \sqrt{\hat{p}(1-\hat{p})/n}$.
- Större stickprov ger smalare intervall. Högre konfidensgrad ger bredare.
- 95 % syftar på **metoden**: 95 av 100 intervall täcker det sanna värdet i längden.
- t-fördelningen kompenserar för att σ skattas med *s*, och det spelar roll för små stickprov.

### Kontrollfrågor
1. Ett 95 % KI för medelinkomsten är [31,2; 35,8] tkr. Vilka av följande påståenden är korrekta?
   a) 95 % av alla personer tjänar mellan 31,2 och 35,8 tkr.
   b) Metoden som gav intervallet fångar det sanna medelvärdet i 95 % av fallen.
   c) Stickprovets medelvärde är 33,5 tkr.
2. Hur stort måste ett stickprov vara för att felmarginalen ska halveras?
3. Varför räcker det inte med ett smalt konfidensintervall för att lita på en undersökning?

<details>
<summary>💡 <b>Svar</b></summary>

1. b och c är korrekta. Intervallet är symmetriskt kring x̄, så x̄ ligger mitt i. Påstående a blandar ihop spridningen
   bland individer med osäkerheten i medelvärdet: intervallet handlar om var *medelvärdet* ligger, inte var enskilda personer ligger.
2. Fyra gånger så stort.
3. Intervallet mäter bara slumpfelet. Bias från bortfall, täckningsfel eller mätfel syns inte i det.

</details>

**Nästa notebook:** [8. Hypotesprövning och p-värden](../08_hypotesprovning/08_hypotesprovning.ipynb)